### General Imports:

In [1]:
import os
import gym
from stable_baselines3.ppo import PPO
from stable_baselines3.ppo.policies import MlpPolicy as MLP_PPO
from netsim.netSimPy import *
from netsim.gym_basic.envs import RMSA_ENV
from netsim.utils import CustomCallback
from stable_baselines3.common.monitor import Monitor
import numpy as np
import tensorflow as tf
import logging
from IPython.display import clear_output
logging.getLogger('tensorflow').setLevel(logging.FATAL)
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import sync_envs_normalization
tf.__version__


ModuleNotFoundError: No module named 'gym'

In [2]:
def warming_lr(initial_lr=1.5e-4, max_warm_lr = 3e-4, final_lr=2e-4, warmup_progress = 0.85):
    def learning_rate_fn(progress):
        if progress >= warmup_progress:
            m = (initial_lr-max_warm_lr)/(1-warmup_progress)
            n = initial_lr - m
            return round(m*progress+n,6)
        else:
            m = (max_warm_lr-final_lr)/ warmup_progress
            return round(m*progress + final_lr,6)
    
    return learning_rate_fn

In [3]:
from typing import Union, Optional

class MyBestCallback(EvalCallback):
    def __init__(
        self,
        eval_env:RMSA_ENV,
        callback_on_new_best = None,
        callback_after_eval = None,
        n_eval_episodes: int = 5,
        eval_freq: int = 10000,
        log_path: Optional[str] = None,
        best_model_save_path: Optional[str] = None,
        deterministic: bool = True,
        render: bool = False,
        verbose: int = 1,
        warn: bool = True,
    ):
        super().__init__(
            eval_env,
            callback_on_new_best,
            callback_after_eval,
            n_eval_episodes,
            eval_freq,
            log_path,
            best_model_save_path,
            deterministic,
            render,
            verbose,
            warn
        )
        self.eval_env = eval_env
        self.best_reward_at = 0

    def _on_step(self) -> bool:
        continue_training = True

        if self.eval_freq > 0 and self.n_calls % self.eval_freq == 0:
            # Sync training and eval env if there is VecNormalize
            if self.model.get_vec_normalize_env() is not None:
                try:
                    sync_envs_normalization(self.training_env, self.eval_env)
                except AttributeError as e:
                    raise AssertionError(
                        "Training and eval env are not wrapped the same way, "
                        "see https://stable-baselines3.readthedocs.io/en/master/guide/callbacks.html#evalcallback "
                        "and warning above."
                    ) from e

            # Reset success rate buffer
            self._is_success_buffer = []
            # self.eval_env.reset(hard_reset=True)
            episode_rewards, episode_lengths = evaluate_policy(
                self.model,
                self.eval_env,
                n_eval_episodes=self.n_eval_episodes,
                render=self.render,
                deterministic=self.deterministic,
                return_episode_rewards=True,
                warn=self.warn,
                callback=self._log_success_callback,
            )

            if self.log_path is not None:
                self.evaluations_timesteps.append(self.num_timesteps)
                self.evaluations_results.append(episode_rewards)
                self.evaluations_length.append(episode_lengths)

                kwargs = {}
                # Save success log if present
                if len(self._is_success_buffer) > 0:
                    self.evaluations_successes.append(self._is_success_buffer)
                    kwargs = dict(successes=self.evaluations_successes)

                np.savez(
                    self.log_path,
                    timesteps=self.evaluations_timesteps,
                    results=self.evaluations_results,
                    ep_lengths=self.evaluations_length,
                    **kwargs,
                )

            mean_reward, std_reward = np.mean(episode_rewards), np.std(episode_rewards)
            mean_ep_length, std_ep_length = np.mean(episode_lengths), np.std(episode_lengths)
            self.last_mean_reward = mean_reward

            if self.verbose >= 1:
                print(f"Eval num_timesteps={self.num_timesteps}, " f"episode_reward={mean_reward:.2f} +/- {std_reward:.2f}")
            # Add to current Logger
            self.logger.record("eval/mean_reward", float(mean_reward))
            self.logger.record("eval/mean_ep_length", mean_ep_length)

            if len(self._is_success_buffer) > 0:
                success_rate = np.mean(self._is_success_buffer)
                if self.verbose >= 1:
                    print(f"Success rate: {100 * success_rate:.2f}%")
                self.logger.record("eval/success_rate", success_rate)

            # Dump log so the evaluation results are printed with the correct timestep
            self.logger.record("time/total_timesteps", self.num_timesteps, exclude="tensorboard")
            self.logger.dump(self.num_timesteps)

            if mean_reward >= self.best_mean_reward:
                self.best_reward_at = self.num_timesteps
                if self.best_model_save_path is not None:
                    self.model.save(os.path.join(self.best_model_save_path, "best_model"))
                self.best_mean_reward = mean_reward
                # Trigger callback on new best model, if needed
                if self.callback_on_new_best is not None:
                    continue_training = self.callback_on_new_best.on_step()
            print(
                f"Best mean reward: {self.best_mean_reward:.2f} at timestep No. {self.best_reward_at}"
            )
            clear_output(wait=True)
            # Trigger callback after every evaluation, if needed
            if self.callback is not None:
                continue_training = continue_training and self._on_event()

        return continue_training

### Training:

In [45]:
from netsim.allocators import sap_ff
# FILES AND DIRECTORIES:
__file__ = 'RMSA.ipynb'
absolutepath = os.path.abspath(__file__)
file_name = os.path.basename(os.path.abspath(__file__)).split('.')[0]
fileDirectory = os.path.dirname(os.path.dirname(os.getcwd())) + "/notebooks/networks"
log_dir=f'./tmp/{file_name}/'
os.makedirs(log_dir, exist_ok=True)
tensorboard_log = f"./tb/{file_name}/nsfnet/"

# SIMULATION PARAMS:
# TOTAL_TIMESTEPS = int(12e6)
TOTAL_TIMESTEPS = int(4e6)
EPISODE_LENGTH = 1000
N_EVALUATION_EPISODES = 80
# EVAL_FREQ = N_EVALUATION_EPISODES * EPISODE_LENGTH 
EVAL_FREQ = 50000
N_BLOCKS = 5
N_PATHS = 3
M_LAMBDA = 5000

network = Network(
    networkFileName = fileDirectory + "/RMSA_NSFNet.json",
    pathsFileName=fileDirectory + "/RMSA_NSFNet_routes.json",
    bitrateFilename=fileDirectory + "/RMSA_NSFNet_bitrate.json",
)
generator = EventsGenerator(mLambda=M_LAMBDA)
env_args = dict(
    network = network,
    eventsGenerator=generator,
    j=N_BLOCKS,
    n_paths=N_PATHS,
    episode_length=EPISODE_LENGTH
)
env: RMSA_ENV = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir)

# MODEL HYPERPARAMETERS:
mArgs = dict(
    # learning_rate=warming_lr(warmup_progress = 0.95),
    # clip_range = 0.1,
    # n_steps= 2048,
    gamma=0.9,
    # ent_coef= 0.01,
    # n_epochs=23,
    # batch_size = 512,
    # ent_coef= 0.0015,
    # n_epochs=27,
    # n_steps= 1024,
    # gamma= 0.95,
    # batch_size = 16,
)
# model = PPO(MLP_PPO, env, verbose=0, seed=3, tensorboard_log=tensorboard_log, **mArgs)
model = PPO(MLP_PPO, env, verbose=0, seed=3, tensorboard_log=tensorboard_log, **mArgs)

# env.setAllocatorFunc(sap_ff)
# callback = CustomCallback(env, eval_freq=EVAL_FREQ, n_eval_episodes=N_EVALUATION_EPISODES, verbose=0, log_path=log_dir)
eval_env:RMSA_ENV = Monitor(gym.make("RMSA_ENV-v0", **env_args), log_dir+"eval_env")
# best_model_save_path = os.path.join(log_dir, "best_model")                     
callback = MyBestCallback(
    eval_env,
    n_eval_episodes=N_EVALUATION_EPISODES, 
    eval_freq=EVAL_FREQ,
    log_path=log_dir, 
    best_model_save_path=log_dir,
    verbose=1)
train_model = model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=callback)



Eval num_timesteps=4000000, episode_reward=980.23 +/- 9.21
Best mean reward: 982.00 at timestep No. 3350000


In [5]:
# Super bueno:
# learning_rate=warming_lr(),
# gamma=0.9,
# M_LAMBDA = 1000
# ------------------------------
# last : 469.50

# mejor segun optuna:

# Best hyperparameters:  {'n_steps': 11, 'learning_rate': 6.035374756982114e-05, 'gamma': 0.9598061435099542, 'batch_size': 9, 'ent_coef': 0.0016629569231729876, 'n_epochs': 23}
# con estos probe y no salio tan mal:
#  {'n_steps': 13, 'learning_rate': 4.3986491084381875e-05, 'gamma': 0.9587320695695378, 'batch_size': 8, 'ent_coef': 0.0015626948449338836, 'n_epochs': 27}
# n_steps= 8192,
# learning_rate=0.00005,
# gamma= 0.95,
# batch_size = 512,
# ent_coef = 0.02, 
# n_epochs = 30

In [47]:
model2 = PPO.load(f"./tmp/{file_name}/best_model.zip")
# Evaluate the policy
policy = model2.policy
env.setAllocatorFunc(None)
env.setLambda(5000)
env.reset(hard_reset=True)

class Test():
    def predict(self, observations, state, episode_start, deterministic):
        return( [4], None)
test = Test()
mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=200, deterministic=True)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=981.11 +/- 8.479852593058443


In [48]:
# Evaluate the policy
policy = model.policy
env.setAllocatorFunc(sap_ff())
env.setLambda(5000)
env.reset(hard_reset=True)
mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=200, deterministic=True)

print(f"mean_reward={mean_reward:.2f} +/- {std_reward}")

mean_reward=983.18 +/- 7.8617809687118605


## Comparison

In [56]:
loads = [1000, 1200, 1400, 1600, 1800, 2000,]
results = {
    "model": [],
    "sap": [],
}
N_EVALUATIONS = 2000000
n_eval_episodes = N_EVALUATIONS / EPISODE_LENGTH
for l in loads:
    env.setLambda(l)
    env.reset(hard_reset=True)
    env.setAllocatorFunc(None)
    mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=n_eval_episodes, deterministic=True)
    print("model",mean_reward)
    results["model"].append(mean_reward)
    
    env.reset(hard_reset=True)
    env.setAllocatorFunc(sap_ff())
    mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=n_eval_episodes, deterministic=True)
    print("sap",mean_reward)
    results["sap"].append(mean_reward)

model 999.964
sap 999.979
model 999.942
sap 999.959
model 999.896
sap 999.926
model 999.747
sap 999.868
model 999.614
sap 999.778
model 999.379
sap 999.634


In [69]:
loads = [1500, 2500, 3000, 3500, 4000, 4500, 6000, 6500 ]
results = {
    "model": [],
    "sap": [],
    "ff": []
}
N_EVALUATIONS = 2000000
n_eval_episodes = N_EVALUATIONS / EPISODE_LENGTH
for l in loads:
    env.setLambda(l)
    env.reset(hard_reset=True)
    env.setAllocatorFunc(sap_ff(1))
    mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=n_eval_episodes, deterministic=True)
    print("ff",mean_reward)
    results["ff"].append(mean_reward)

ff 999.542
ff 995.754
ff 991.71
ff 986.037
ff 979.272
ff 971.622
ff 946.177
ff 937.653


In [58]:
loads = [1500, 2500, 3000, 3500, 4000, 4500, 6000, 6500 ]
results = {
    "model": [],
    "sap": [],
}
N_EVALUATIONS = 2000000
n_eval_episodes = N_EVALUATIONS / EPISODE_LENGTH
for l in loads:
    env.setLambda(l)
    env.reset(hard_reset=True)
    env.setAllocatorFunc(None)
    mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=n_eval_episodes, deterministic=True)
    print("model",mean_reward)
    results["model"].append(mean_reward)
    
    env.reset(hard_reset=True)
    env.setAllocatorFunc(sap_ff())
    mean_reward, std_reward = evaluate_policy(policy, env, n_eval_episodes=n_eval_episodes, deterministic=True)
    print("sap",mean_reward)
    results["sap"].append(mean_reward)

model 999.845
sap 999.902
model 998.502
sap 998.975
model 996.757
sap 997.664
model 993.953
sap 995.4
model 989.874
sap 992.127
model 984.841
sap 987.781
model 964.284
sap 970.133
model 956.533
sap 963.177


### Blocking Prob

In [62]:
loads = [1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500 ]

model = [999.964, 999.845, 999.379, 998.502, 996.757, 993.953, 989.874, 984.841]
sap = [999.979, 999.902, 999.634, 998.975, 997.664, 995.400, 992.127, 987.781]

model_bp = []
sap_bp = []
for (m,s) in zip(model,sap):
    model_bp.append(round((1000 - m) / 2000,7) )
    sap_bp.append(round((1000 - s) / 2000,7))
print(model_bp, sap_bp)

[1.8e-05, 7.75e-05, 0.0003105, 0.000749, 0.0016215, 0.0030235, 0.005063, 0.0075795] [1.05e-05, 4.9e-05, 0.000183, 0.0005125, 0.001168, 0.0023, 0.0039365, 0.0061095]


In [72]:
ff = [999.542,995.754,991.71,986.037, 979.272, 971.622, 946.177, 937.653]
ffs = []
for m in ff:
    ffs.append(round((1000 - m) / 2000,7) )
print(ffs)

[0.000229, 0.002123, 0.004145, 0.0069815, 0.010364, 0.014189, 0.0269115, 0.0311735]


In [49]:
def generate_trajectories(n_timesteps, model, env, only_accepted = True):
    trajectories = []
    timesteps_count = 0
    state = None
    done = False
    observation = env.reset()
    
    while (timesteps_count < n_timesteps):
        act, state = model.predict(
                            observation,
                            state=state,
                            episode_start=done,
                            deterministic=True,
                        )
        traj = {
            'obs': observation,
            'action': act
        }
        observation, reward, done, info = env.step(act)
        traj['reward'] = reward
        if only_accepted:
            if reward > 0:
                trajectories.append(traj)
                timesteps_count+=1
        else:
            trajectories.append(traj)
            timesteps_count+=1
        if done:
            env.reset()
    return trajectories

def flatten_trajectories(traj):
    obs = [t['obs'] for t in traj]
    acts = [t['action'] for t in traj]
    rewards = [t['reward'] for t in traj]
    return obs, acts, rewards

In [52]:
traj = generate_trajectories(100000, policy, env)


In [53]:
actions = [0]*15
for t in traj:
    act = t['action']
    actions[act] = actions[act] + 1
    # print(act)

print(actions)

[64436, 3646, 995, 12, 5, 14435, 6283, 219, 0, 22, 6482, 958, 1935, 572, 0]
